# Approach 3 — Qwen3-VL with LoRA

Notebook 09 is the single maintained training path for the Qwen3-VL baseline and QLoRA adapter. This companion notebook reloads the saved results and allows report review; it intentionally does not duplicate training or start an unbounded run.

## Before running

Use the local project checkout. Notebook 09 saves validation metrics and predictions under `QWEN_SAVE_DIR` (default: `<project>/qwen3vl_cxr`). The old 24-case test predictions remain available for audit, but they were exposed earlier and are not valid final held-out evidence. BLEU and ROUGE-L measure text overlap, not medical accuracy.


In [1]:
from pathlib import Path
import os, sys
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((p.resolve() for p in CANDIDATES if (p / "src" / "config.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the cxr_second_reader project folder in VS Code/Jupyter.")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Local project folder:", REPO_ROOT)


Project: D:\cxr_second_reader


In [2]:
import json
import pandas as pd
from IPython.display import display

RESULTS = Path(os.environ.get("QWEN_SAVE_DIR", REPO_ROOT / "qwen3vl_cxr")).expanduser()
validation_files = [
    "metrics_qwen_zero_shot_validation.json", "metrics_qwen_lora_validation_current.json",
    "eval_qwen_zero_shot_validation.csv", "eval_qwen_lora_validation_current.csv",
]
legacy_files = [
    "metrics_qwen_zero_shot.json", "metrics_qwen_lora_finetuned.json",
    "eval_qwen_zero_shot.csv", "eval_qwen_lora_finetuned.csv",
]
if all((RESULTS / name).is_file() for name in validation_files):
    selected_files = validation_files
    print("Displaying validation-development results.")
elif all((RESULTS / name).is_file() for name in legacy_files):
    selected_files = legacy_files
    print("WARNING: these are legacy 24-case results from before the step-920 checkpoint; not final metrics.")
else:
    raise FileNotFoundError(f"No complete local Qwen result pair found under {RESULTS}. Run notebook 09 first.")
print("Results folder:", RESULTS)
base = json.loads((RESULTS / selected_files[0]).read_text(encoding="utf-8"))
ft = json.loads((RESULTS / selected_files[1]).read_text(encoding="utf-8"))
display(pd.DataFrame([{"stage": "Qwen zero-shot", **base}, {"stage": "Qwen QLoRA", **ft}]))


Results folder: D:\cxr_second_reader\qwen3vl_cxr
            stage    rougeL      bleu   n
0  Qwen zero-shot  0.187889  0.026596  24
1      Qwen QLoRA  0.331241  0.069996  24


In [3]:
zero = pd.read_csv(RESULTS / selected_files[2])
finetuned = pd.read_csv(RESULTS / selected_files[3])
print("Rows:", len(zero), len(finetuned))
print("Matching study IDs:", set(zero["uid"].astype(str)) == set(finetuned["uid"].astype(str)))
review = finetuned[[c for c in ["uid", "filename", "prediction", "reference", "rougeL", "bleu"] if c in finetuned.columns]]
display(review.head(5))
print("Review generated findings against images; text metrics cannot identify unsupported or omitted clinical findings.")


Rows: 24 24
Matching study IDs: True
    uid                   filename  ...    rougeL      bleu
0  2924  2924_IM-1327-1001.dcm.png  ...  0.313253  0.030792
1  1410  1410_IM-0260-1001.dcm.png  ...  0.356164  0.049788
2  3062  3062_IM-1428-1001.dcm.png  ...  0.326531  0.034173
3   100   100_IM-0002-1001.dcm.png  ...  0.484848  0.110628
4  1219  1219_IM-0146-1001.dcm.png  ...  0.360000  0.075654

[5 rows x 6 columns]
Review generated findings against each image; text metrics cannot identify unsupported or omitted clinical findings.


## Continue training

For more training, return to `09_qwen3vl_cxr.ipynb`, restore the QLoRA checkpoint and optimizer state, and run another saved training chunk. After choosing a checkpoint with validation data, rerun the notebook's final adapter-save and held-out evaluation cells. Do not repeatedly tune against the test split.